# Combine 3 Food Datasets (Malaysia Food 11, MYFood 11 COCO, MyFood50)

**Run all cells** to get **one merged dataset** containing all images from the 3 sources.

- **No train/test folders**: All images go into folders by food item only. The CSV keeps a `split` column (train/test/validation) so you can split later (e.g. with sklearn).
- **CNN-ready**: All images are resized to a **standard size** (e.g. 224×224) and saved as JPEG when copied.
- **Clean labels**: Automatic normalization and synonym mapping so you don't rename manually.
- **Safety**: Path validation, safe JSON/zip handling.

## 1. Imports and paths

In [1]:
import json
import os
import re
import shutil
from pathlib import Path
from collections import Counter, defaultdict
import zipfile

import io
import pandas as pd
from PIL import Image

# Base folder = folder containing this notebook
BASE = Path(".").resolve()

MALAYSIA_11 = BASE / "Malaysia Food 11"
COCO_11 = BASE / "MYFood 11.coco"
MYFOOD50 = BASE / "MyFood50"
OUT_DIR = BASE / "combined_dataset"
OUT_DIR.mkdir(exist_ok=True)

ALLOWED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Standard image size for CNN (e.g. ResNet, VGG use 224; change if needed)
IMAGE_SIZE = (224, 224)
print("BASE:", BASE)
print("OUT_DIR:", OUT_DIR)
print("IMAGE_SIZE (for CNN):", IMAGE_SIZE)

BASE: C:\Users\MSI GF63\Desktop\DatasetRasaRight
OUT_DIR: C:\Users\MSI GF63\Desktop\DatasetRasaRight\combined_dataset
IMAGE_SIZE (for CNN): (224, 224)


## 2. Safety: path validation and label normalization

In [2]:
def safe_resolve(base: Path, *parts: str):
    """Resolve path under base; return None if outside base (path traversal)."""
    try:
        p = (base / os.path.join(*parts)).resolve()
        p = p.resolve()
        base_resolved = base.resolve()
        if os.path.commonpath([p, base_resolved]) != str(base_resolved):
            return None
        return p
    except (ValueError, OSError):
        return None


def normalize_label(raw: str) -> str:
    """Snake_case / hyphen → Title Case; strip extra spaces."""
    s = re.sub(r"[_-]", " ", str(raw).strip())
    return " ".join(w.capitalize() for w in s.split())


# Map raw labels to a single canonical name (avoids manual renaming)
LABEL_ALIASES = {
    "burger": "Burger",
    "hamburger": "Burger",
    "nasi goreng": "Nasi Goreng",
    "fried rice": "Fried Rice",
    "nasi lemak": "Nasi Lemak",
    "roti canai": "Roti Canai",
    "kaya toast": "Kaya Toast",
    "fish and chips": "Fish And Chips",
    "fried noodles": "Fried Noodles",
    "curry puff": "Curry Puff",
    "french fries": "French Fries",
    "ayam goreng": "Fried Chicken",
    "fried chicken": "Fried Chicken",
    "boiled eggs": "Boiled Eggs",
    "mei goreng": "Mei Goreng",
    "rice": "Rice",
    "mixed rice": "Mixed Rice",
    "popiah": "Popiah",
    "satay": "Satay",
    "laksa": "Laksa",
    "pizza": "Pizza",
    "steak": "Steak",
}


def to_canonical_label(raw: str) -> str:
    """Normalize then map via alias so datasets align without manual rename."""
    n = normalize_label(raw)
    key = n.lower()
    return LABEL_ALIASES.get(key, n)

## 3. Load Malaysia Food 11 (folder-based)

In [3]:
def load_malaysia_11(base: Path):
    rows = []
    if not base.exists():
        return rows
    for folder in base.iterdir():
        if not folder.is_dir():
            continue
        label_raw = folder.name
        for f in folder.iterdir():
            if f.suffix.lower() not in ALLOWED_EXT:
                continue
            res = safe_resolve(base, folder.name, f.name)
            if res is None or not res.exists():
                continue
            rows.append({
                "path": str(res),
                "path_type": "file",
                "raw_label": label_raw,
                "source": "Malaysia_Food_11",
                "split": "train",
            })
    return rows

malaysia_rows = load_malaysia_11(MALAYSIA_11)
print(f"Malaysia Food 11: {len(malaysia_rows)} images")

Malaysia Food 11: 11000 images


## 4. Load MYFood 11 (COCO)

In [4]:
def load_coco_11(base: Path):
    rows = []
    ann_path = base / "train" / "_annotations.coco.json"
    train_dir = base / "train"
    if not ann_path.exists() or not train_dir.is_dir():
        return rows
    with open(ann_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    id_to_cat = {c["id"]: c["name"] for c in data.get("categories", []) if c["id"] != 0}
    im_cats = defaultdict(Counter)
    for ann in data.get("annotations", []):
        im_id = ann.get("image_id")
        cat_id = ann.get("category_id")
        if im_id is not None and cat_id in id_to_cat:
            im_cats[im_id][id_to_cat[cat_id]] += 1
    for im in data.get("images", []):
        im_id = im["id"]
        fname = im.get("file_name", "")
        if not fname:
            continue
        res = safe_resolve(train_dir, fname)
        if res is None or not res.exists():
            continue
        label_raw = im_cats[im_id].most_common(1)[0][0] if im_cats[im_id] else (id_to_cat.get(1) or "Unknown")
        rows.append({
            "path": str(res),
            "path_type": "file",
            "raw_label": label_raw,
            "source": "MYFood_11_COCO",
            "split": "train",
        })
    return rows

coco_rows = load_coco_11(COCO_11)
print(f"MYFood 11 COCO: {len(coco_rows)} images")

MYFood 11 COCO: 750 images


In [5]:
# COCO loader is defined in the cell above; coco_rows is ready for merge.

## 5. Load MyFood50 (from zips, no extract — low memory)

In [6]:
def load_myfood50_zips(base: Path):
    rows = []
    split_to_zip = {
        "train": base / "training.zip",
        "test": base / "test.zip",
        "validation": base / "validation.zip",
    }
    for split, zpath in split_to_zip.items():
        if not zpath.exists():
            continue
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                for name in zf.namelist():
                    if name.endswith("/"):
                        continue
                    p = Path(name)
                    if p.suffix.lower() not in ALLOWED_EXT:
                        continue
                    # Label = parent folder name (e.g. training/Nasi lemak/0.jpg -> Nasi lemak)
                    parts = name.replace("\\", "/").split("/")
                    if len(parts) >= 2:
                        label_raw = parts[-2]
                    else:
                        label_raw = "Unknown"
                    # Store as zip_path|member for loader
                    rows.append({
                        "path": f"{zpath}|{name}",
                        "path_type": "zip",
                        "raw_label": label_raw,
                        "source": "MyFood50",
                        "split": split,
                    })
        except (zipfile.BadZipFile, OSError) as e:
            print(f"Warning: {zpath} - {e}")
    return rows

myfood50_rows = load_myfood50_zips(MYFOOD50)
print(f"MyFood50: {len(myfood50_rows)} images")

MyFood50: 42426 images


## 6. Merge and build unified labels

In [7]:
all_rows = malaysia_rows + coco_rows + myfood50_rows
df = pd.DataFrame(all_rows)
df["label"] = df["raw_label"].map(to_canonical_label)

labels_sorted = sorted(df["label"].unique())
label2id = {name: i for i, name in enumerate(labels_sorted)}
id2label = {i: name for name, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)
print(f"Total samples: {len(df)}")
print(f"Unique labels: {len(label2id)}")
print("\nLabel list (first 25):", labels_sorted[:25])

Total samples: 54176
Unique labels: 65

Label list (first 25): ['Apam Balik', 'Bak Kut Teh', 'Beef Rendang', 'Bibimbap', 'Biryani', 'Boiled Eggs', 'Burger', 'Cendol', 'Char Kuey Teow', 'Char Siew', 'Cheesecake', 'Chicken Curry', 'Chicken Rendang', 'Chicken Rice', 'Chili Crab', 'Chocolate Cake', 'Cup Cakes', 'Curry Mee', 'Curry Puff', 'Donuts', 'Dumplings', 'Egg Tart', 'Fish And Chips', 'French Fries', 'Fried Chicken']


## 7. Copy all images into one folder (by food item)

In [8]:
def sanitize_foldername(name):
    """Safe folder name for Windows: no \ / : * ? \" < > |"""
    s = re.sub(r'[<>:"/\\|?*]+', "_", str(name).strip())
    return s.strip("._") or "Unknown"


def load_image_for_copy(row, base):
    """Load image as PIL Image from file or zip. Returns None on error."""
    try:
        if row["path_type"] == "file":
            src = Path(row["path"])
            if not src.is_absolute():
                src = base / row["path"]
            if not src.exists():
                return None
            return Image.open(src).convert("RGB")
        else:
            if "|" not in row["path"]:
                return None
            zpath, member = row["path"].split("|", 1)
            zpath = Path(zpath)
            if not zpath.is_absolute():
                zpath = base / zpath
            if not zpath.exists():
                return None
            with zipfile.ZipFile(zpath, "r") as zf:
                data = zf.read(member)
            return Image.open(io.BytesIO(data)).convert("RGB")
    except (OSError, zipfile.BadZipFile, KeyError):
        return None


def copy_all_into_one_folder(df, out_dir, base, image_size):
    """Copy every image into out_dir/<label>/<label>_<idx>.jpg, resized to image_size for CNN."""
    out_dir = Path(out_dir)
    base = Path(base)
    counters = defaultdict(int)
    merged_paths = []
    for idx, row in df.iterrows():
        label = row["label"]
        folder = out_dir / sanitize_foldername(label)
        folder.mkdir(parents=True, exist_ok=True)
        counters[label] += 1
        num = counters[label]
        safe_label = sanitize_foldername(label)
        dest = folder / f"{safe_label}_{num:05d}.jpg"
        try:
            img = load_image_for_copy(row, base)
            if img is None:
                merged_paths.append(None)
                continue
            img = img.resize(image_size, getattr(Image, "Resampling", Image).LANCZOS)
            img.save(dest, "JPEG", quality=92)
            merged_paths.append(str(dest))
        except (OSError, zipfile.BadZipFile, KeyError) as e:
            merged_paths.append(None)
            if (idx + 1) % 500 == 0 or idx == 0:
                print(f"  Skip row {idx}: {e}")
    df = df.copy()
    df["merged_path"] = merged_paths
    return df


print("Copying and resizing all images to", IMAGE_SIZE, "into", OUT_DIR, "by food item...")
df = copy_all_into_one_folder(df, OUT_DIR, BASE, IMAGE_SIZE)
failed = df["merged_path"].isna().sum()
print(f"Done. {len(df) - failed} copied, {failed} skipped (missing/error).")
df = df[df["merged_path"].notna()].copy()
print(f"Final rows: {len(df)}")

Copying and resizing all images to (224, 224) into C:\Users\MSI GF63\Desktop\DatasetRasaRight\combined_dataset by food item...


<>:2: SyntaxWarning: invalid escape sequence '\ '
<>:2: SyntaxWarning: invalid escape sequence '\ '
C:\Users\MSI GF63\AppData\Local\Temp\ipykernel_8356\2882421478.py:2: SyntaxWarning: invalid escape sequence '\ '
  """Safe folder name for Windows: no \ / : * ? \" < > |"""
c:\Users\MSI GF63\anaconda3\Lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Done. 54175 copied, 1 skipped (missing/error).
Final rows: 54175


## 8. Save combined index (single folder paths)

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified 80% train, 20% rest
train_idx, temp_idx = train_test_split(
    df.index, test_size=0.2, stratify=df["label_id"], random_state=42
)
# Split the 20% into 50-50 → 10% validation, 10% test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=df.loc[temp_idx, "label_id"], random_state=42
)
df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"
print("Split counts:", df["split"].value_counts().to_dict())
print("Fractions: train {:.1%}, validation {:.1%}, test {:.1%}".format(
    (df["split"] == "train").mean(),
    (df["split"] == "validation").mean(),
    (df["split"] == "test").mean(),
))

In [9]:
# Save index pointing to the merged folder (one place for all images)
df_out = df[["merged_path", "label", "label_id", "source", "split"]].rename(columns={"merged_path": "path"})
df_out["path_type"] = "file"
df_out.to_csv(OUT_DIR / "combined_index.csv", index=False)
with open(OUT_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(id2label, f, indent=2)
with open(OUT_DIR / "label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, indent=2)
print("Saved:", OUT_DIR / "combined_index.csv", OUT_DIR / "labels.json", OUT_DIR / "label2id.json")
print("Folder structure:", OUT_DIR)
for d in sorted(OUT_DIR.iterdir())[:15]:
    if d.is_dir():
        n = len(list(d.glob("*")))
        print(f"  {d.name}/  ({n} images)")

Saved: C:\Users\MSI GF63\Desktop\DatasetRasaRight\combined_dataset\combined_index.csv C:\Users\MSI GF63\Desktop\DatasetRasaRight\combined_dataset\labels.json C:\Users\MSI GF63\Desktop\DatasetRasaRight\combined_dataset\label2id.json
Folder structure: C:\Users\MSI GF63\Desktop\DatasetRasaRight\combined_dataset
  Apam Balik/  (190 images)
  Bak Kut Teh/  (222 images)
  Beef Rendang/  (156 images)
  Bibimbap/  (807 images)
  Biryani/  (582 images)
  Boiled Eggs/  (54 images)
  Burger/  (2064 images)
  Cendol/  (188 images)
  Char Kuey Teow/  (1096 images)
  Char Siew/  (251 images)
  Cheesecake/  (1000 images)
  Chicken Curry/  (1000 images)
  Chicken Rendang/  (173 images)
  Chicken Rice/  (721 images)
  Chili Crab/  (204 images)


## 9. Safe image loader (file or zip)

In [10]:
def load_image_from_row(row, base=None):
    """Load image for one row. Returns PIL Image or None. Safe for path and zip."""
    try:
        from PIL import Image
        import io
    except ImportError:
        return None
    path = row.get("path")
    path_type = row.get("path_type", "file")
    if path_type == "zip":
        if "|" not in path:
            return None
        zpath, member = path.split("|", 1)
        zpath = Path(zpath)
        if base and not zpath.is_absolute():
            zpath = base / zpath
        if not zpath.exists():
            return None
        with zipfile.ZipFile(zpath, "r") as zf:
            data = zf.read(member)
        return Image.open(io.BytesIO(data)).convert("RGB")
    else:
        p = Path(path)
        if base and not p.is_absolute():
            p = base / path
        if not p.exists():
            return None
        return Image.open(p).convert("RGB")


# Quick check: load one from merged folder (path = merged_path, path_type = file)
for src in ["Malaysia_Food_11", "MYFood_11_COCO", "MyFood50"]:
    sub = df[df["source"] == src]
    if len(sub) > 0:
        r = sub.iloc[0].to_dict()
        r["path"] = r.get("merged_path") or r["path"]
        r["path_type"] = "file"
        img = load_image_from_row(r, base=BASE)
        print(f"{src}: {"OK" if img is not None else "FAIL"}")

Malaysia_Food_11: OK
MYFood_11_COCO: OK
MyFood50: OK


## 10. Optional: PyTorch Dataset (load from merged folder)

In [11]:
class CombinedFoodDataset:
    def __init__(self, index_csv, base_path, transform=None, split_filter=None):
        self.base = Path(base_path)
        self.df = pd.read_csv(index_csv)
        if split_filter:
            self.df = self.df[self.df["split"].isin(split_filter)]
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i].to_dict()
        img = load_image_from_row(row, base=self.base)
        if img is None:
            raise FileNotFoundError(row.get("path", ""))
        label_id = int(row["label_id"])
        if self.transform:
            img = self.transform(img)
        return img, label_id


# Example usage (uncomment if you have torch + PIL):
# ds = CombinedFoodDataset(OUT_DIR / "combined_index.csv", BASE, split_filter=["train"])
# img, lab = ds[0]
print("CombinedFoodDataset defined. Use: ds = CombinedFoodDataset(OUT_DIR / 'combined_index.csv', BASE)")

CombinedFoodDataset defined. Use: ds = CombinedFoodDataset(OUT_DIR / 'combined_index.csv', BASE)
